# Assembly Graph Twin — Google Colab benchmark runner

This notebook runs the repository’s existing benchmark CLI in a hosted Python runtime. It does **not** reimplement the graph or change the CLI. Its optional PyTorch backend sends k-mer window encoding, per-read edge deduplication, and support aggregation to the selected tensor device.

**GPU scope:** CUDA performs k-mer encoding, per-read edge deduplication, edge/node support aggregation, and adjacency-index construction. Python string-key graph creation, graph searches, diagnostics, reference placement, and path labeling remain CPU work. The notebook streams phase progress so a long stage is visible. No exponential speedup is promised, and GPU memory does not replace system RAM. Reference placement and breadth are estimates, not production alignments.

## Before running

1. In Google Drive, copy the project folder `assembly_graph_twin` so it contains `assembly_graph_twin.py`, `benchmark.py`, `data/reads/`, and `data/references/`. Large inputs can stay in Drive; do not embed them in the notebook.
2. Open this notebook in Colab.
3. Run the setup and configuration cells. Edit the paths in the configuration cell if your Drive folders differ.
4. Start with `MAX_READS = 50`. If it completes and memory usage is acceptable, run the separate 250-read cell.

You can also use local files already uploaded to `/content`; set `PROJECT_DIR`, `READS_PATH`, and `REFERENCE_PATH` accordingly.


In [ ]:
# Colab setup: mount Google Drive when running in Colab. This is a no-op on local Jupyter.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Google Colab not detected; configure paths for this Jupyter environment.")


## Configure project and input paths

Change only the paths below to point at your copy of the repository and the chosen read/reference files. The defaults match the repository’s bundled *S. aureus* example data when the project folder is copied to `MyDrive/assembly_graph_twin`.

For GPU graph construction, select **Runtime → Change runtime type → T4 GPU** in Colab. The configuration below requests CUDA explicitly and stops with a clear error if CUDA is unavailable. Change `GRAPH_BACKEND` to `"cpu"` and `DEVICE` to `"cpu"` to use the unchanged CPU path.


In [ ]:
from pathlib import Path
import os
import sys

# EDIT THIS if you placed the project somewhere else in Google Drive.
PROJECT_DIR = Path("/content/drive/MyDrive/assembly_graph_twin")

# These may be absolute paths or paths relative to PROJECT_DIR.
READS_PATH = PROJECT_DIR / "data" / "reads" / "SRR31691735_1.fastq.gz"
REFERENCE_PATH = PROJECT_DIR / "data" / "references" / "sa_reference.fa"
OUTPUT_DIR = PROJECT_DIR / "reports" / "benchmarks" / "colab"

# Start small. Increase MAX_READS in the later cell after inspecting this run.
K = 15
MAX_READS = 50
SEED = 1
DATASET_ID = "sa-atcc43300-colab-k15-smoke"
BUBBLE_DEPTH = 3
BUBBLE_SEARCH_NODE_CAP = 64
MIN_REFERENCE_IDENTITY = 0.90
STRAND_MODE = "bidirected"
GRAPH_BACKEND = "torch"  # "torch" uses PyTorch k-mer encoding; "cpu" uses the default implementation.
DEVICE = "cuda"  # Colab T4. Use "cpu" for the CPU backend/fallback.

# Optional: if True, load the report instead of rerunning when it already exists.
# Leave False when you intentionally want a fresh run.
LOAD_EXISTING_REPORT = False

PROJECT_DIR = Path(PROJECT_DIR).expanduser()
READS_PATH = Path(READS_PATH).expanduser()
REFERENCE_PATH = Path(REFERENCE_PATH).expanduser()
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser()

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_DIR}\n"
        "Copy the assembly_graph_twin project folder to Google Drive, then update PROJECT_DIR."
    )
for required_file in (PROJECT_DIR / "assembly_graph_twin.py", PROJECT_DIR / "benchmark.py"):
    if not required_file.is_file():
        raise FileNotFoundError(f"Expected project file is missing: {required_file}")
for input_file in (READS_PATH, REFERENCE_PATH):
    if not input_file.is_file():
        raise FileNotFoundError(
            f"Input file not found: {input_file}\n"
            "Update READS_PATH or REFERENCE_PATH in the configuration cell."
        )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = OUTPUT_DIR / f"{DATASET_ID}-k{K}-{MAX_READS}.json"
print(f"Project:   {PROJECT_DIR}")
print(f"Reads:     {READS_PATH} ({READS_PATH.stat().st_size / (1024**2):.1f} MiB compressed/on disk)")
print(f"Reference: {REFERENCE_PATH} ({REFERENCE_PATH.stat().st_size / (1024**2):.1f} MiB on disk)")
print(f"Output:    {REPORT_PATH}")

if GRAPH_BACKEND == "torch":
    try:
        import torch
    except ImportError as error:
        raise RuntimeError(
            "PyTorch is not installed in this runtime. Colab usually includes it; "
            "otherwise install the optional dependency with %pip install torch."
        ) from error
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable. In Colab choose Runtime > Change runtime type > T4 GPU, "
            "reconnect, and rerun this cell."
        )
    actual = torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE
    print(f"Graph construction backend: {GRAPH_BACKEND}; device: {actual}")


In [ ]:
# Detect a stale or mixed project copy before starting an expensive benchmark.
import json
app_source = (PROJECT_DIR / "assembly_graph_twin.py").read_text(encoding="utf-8")
benchmark_source = (PROJECT_DIR / "benchmark.py").read_text(encoding="utf-8")
gpu_source_path = PROJECT_DIR / "gpu_backend.py"
protocol_path = PROJECT_DIR / "config" / "benchmark_protocol.json"
protocol = json.loads(protocol_path.read_text(encoding="utf-8")) if protocol_path.is_file() else {}
required_markers = {
    "assembly_graph_twin.py GPU CLI and aggregated graph support": (
        "args.graph_backend" in app_source
        and "--no-python-memory-tracing" in app_source
        and "backend: str = \"cpu\"" in app_source
        and "def add_aggregated_edge(" in app_source
    ),
    "benchmark.py backend integration": "backend=graph_backend, device=device" in benchmark_source,
    "gpu_backend.py present": gpu_source_path.is_file(),
    "benchmark protocol v3.3": protocol.get("protocol_version") == "3.3",
    "GPU implementation markers": gpu_source_path.is_file() and "torch.unique" in gpu_source_path.read_text(encoding="utf-8") and "_finalize_adjacency_on_device" in gpu_source_path.read_text(encoding="utf-8"),
}
for label, present in required_markers.items():
    print(f"{label}: {'OK' if present else 'MISSING/OLD'}")
if not all(required_markers.values()):
    raise RuntimeError(
        "The Drive project files are stale or mixed versions. Replace assembly_graph_twin.py, "
        "benchmark.py, and gpu_backend.py together from the latest repository copy, then rerun this notebook."
    )


## Run the benchmark

This cell invokes the same `assembly_graph_twin.py benchmark` command used locally. The JSON report is saved under `reports/benchmarks/colab/`. The notebook streams CLI progress messages while the job runs. The seeded bottom-k sampler makes the selected subset repeatable for the same input, seed, and size.


In [ ]:
import json
import subprocess
from IPython.display import display


def benchmark_command(k: int, max_reads: int, dataset_id: str, report_path: Path) -> list[str]:
    """Build the existing project CLI command; no alternate graph implementation."""
    return [
        sys.executable, str(PROJECT_DIR / "assembly_graph_twin.py"), "benchmark",
        "--reads", str(READS_PATH),
        "--reference", str(REFERENCE_PATH),
        "--k", str(k),
        "--max-reads", str(max_reads),
        "--seed", str(SEED),
        "--dataset-id", dataset_id,
        "--bubble-depth", str(BUBBLE_DEPTH),
        "--bubble-search-node-cap", str(BUBBLE_SEARCH_NODE_CAP),
        "--min-reference-identity", str(MIN_REFERENCE_IDENTITY),
        "--strand-mode", STRAND_MODE,
        "--graph-backend", GRAPH_BACKEND,
        "--device", DEVICE,
        "--no-python-memory-tracing",
        "--output", str(report_path),
    ]


def run_and_summarize(k: int, max_reads: int, dataset_id: str, report_path: Path, *, load_existing: bool = False):
    if load_existing and report_path.is_file():
        print(f"Loading existing report: {report_path}")
        report = json.loads(report_path.read_text(encoding="utf-8"))
    else:
        command = benchmark_command(k, max_reads, dataset_id, report_path)
        print("Running CLI command:")
        print(subprocess.list2cmdline(command))
        process = subprocess.Popen(
            command,
            cwd=PROJECT_DIR,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        assert process.stdout is not None
        for output_line in process.stdout:
            print(output_line, end="", flush=True)
        return_code = process.wait()
        if return_code != 0:
            raise RuntimeError(f"Benchmark exited with code {return_code}")
        report = json.loads(report_path.read_text(encoding="utf-8"))

    metrics = report.get("metrics", {})
    inputs = report.get("inputs", {})
    protocol = report.get("protocol", {})
    reference = report.get("reference_validation", {})
    if GRAPH_BACKEND == "torch" and DEVICE == "cuda":
        if protocol.get("graph_backend") != "torch" or protocol.get("graph_device") != "cuda":
            raise RuntimeError("Report does not confirm the requested PyTorch CUDA graph backend.")
        if metrics.get("gpu_peak_allocated_bytes", 0) <= 0:
            raise RuntimeError("CUDA was requested but the run recorded no CUDA allocation.")
        print(f"CUDA confirmed: {metrics['gpu_peak_allocated_bytes'] / (1024**2):.1f} MiB peak allocated")
    summary = {
        "k": protocol.get("k", k),
        "graph_backend": protocol.get("graph_backend"),
        "graph_device": protocol.get("graph_device"),
        "selected_reads": inputs.get("selected_read_count", max_reads),
        "candidate_count": metrics.get("candidate_count"),
        "uniquely_labeled_candidates": metrics.get("uniquely_labeled_candidate_count"),
        "resolved_accuracy": metrics.get("accuracy_on_resolved_labeled"),
        "runtime_seconds": metrics.get("runtime_seconds"),
        "graph_construction_seconds": metrics.get("graph_construction_seconds"),
        "graph_diagnostics_seconds": metrics.get("graph_diagnostics_seconds"),
        "reference_validation_seconds": metrics.get("reference_validation_seconds"),
        "gpu_peak_allocated_GiB": round(metrics.get("gpu_peak_allocated_bytes", 0) / (1024**3), 3),
        "gpu_peak_reserved_GiB": round(metrics.get("gpu_peak_reserved_bytes", 0) / (1024**3), 3),
        "python_tracemalloc_peak_GiB": (round(metrics["python_tracemalloc_peak_bytes"] / (1024**3), 3) if metrics.get("python_tracemalloc_peak_bytes") is not None else "disabled for runtime"),
        "reference_placed_reads": reference.get("reference_placed_read_count"),
        "estimated_reference_breadth": reference.get("estimated_reference_breadth"),
        "graph_edge_support_fraction": reference.get("graph_edge_support_fraction"),
        "report": str(report_path),
    }
    display(summary)
    graph_metrics = inputs.get("graph_metrics")
    if graph_metrics:
        display({"graph_metrics": graph_metrics})
    diagnostics = inputs.get("diagnostic_summary")
    if diagnostics:
        # Keep the notebook output compact; the complete diagnostic evidence remains in JSON.
        compact_diagnostics = {
            key: value for key, value in diagnostics.items() if key != "examples"
        }
        display({"diagnostics": compact_diagnostics})
        if diagnostics.get("examples", {}).get("bubble_search_limited"):
            display({"example_bounded_search": diagnostics["examples"]["bubble_search_limited"][0]})
    return report

REPORT = run_and_summarize(
    K, MAX_READS, DATASET_ID, REPORT_PATH, load_existing=LOAD_EXISTING_REPORT
)


## Optional larger run: 250 reads

Run this only after the smoke run finishes and you have checked the reported peak memory. The graph can use substantial memory; a hosted runtime may still run out. This cell uses the same seed and settings, so the 50-read subset is nested within the 250-read subset.

The smaller `k` values are exploratory and are not the frozen protocol default (`k=31`). Compare their outputs as experiments, not as apples-to-apples changes in biological truth.


In [ ]:
# Set K_LARGE to 15 or 10 for the exploratory run you want to try.
K_LARGE = 15
MAX_READS_LARGE = 250
DATASET_ID_LARGE = f"sa-atcc43300-colab-k{K_LARGE}-250"
REPORT_PATH_LARGE = OUTPUT_DIR / f"{DATASET_ID_LARGE}.json"

REPORT_LARGE = run_and_summarize(
    K_LARGE, MAX_READS_LARGE, DATASET_ID_LARGE, REPORT_PATH_LARGE,
    load_existing=False,
)


## Interpret the results carefully

- `candidate_count = 0` means the bounded detector did not report a complete bubble under these search settings. It does not prove the genome has no variants.
- Reference placement and breadth are anchor-based estimates, not gapped alignments or a consensus assembly.
- A zero-candidate run has no bubble accuracy score to interpret.
- The graph is not a finished assembly: this workflow does not emit contigs.
- The report records the backend and device used. CUDA k-mer encoding, per-read deduplication, support aggregation, and adjacency-index construction are GPU stages. Python graph-key creation, graph traversal, diagnostics, and reference checks remain CPU work.

The same workflow remains available from a local Command Prompt. From the repository root, for example:

```cmd
python assembly_graph_twin\assembly_graph_twin.py benchmark --reads assembly_graph_twin\data\reads\SRR31691735_1.fastq.gz --reference assembly_graph_twin\data\references\sa_reference.fa --k 15 --max-reads 50 --seed 1 --dataset-id sa-atcc43300-k15-smoke --output assembly_graph_twin\reports\benchmarks\sa-atcc43300-k15-smoke.json
```
